# Ekegusii PSA translator — run the demo from Colab

Runs the same service as `serve/app.py` on Colab's free GPU and puts it behind a
public URL. Use this when the training node is gone, or on presentation day from
any machine.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. On CPU it still
works, but a sentence takes several seconds instead of about one.

Roughly five minutes end to end, most of it downloading 2.4 GB of weights.

> The URL this prints is temporary — it dies when the Colab session ends, and
> Colab disconnects after a period of inactivity. Start it before you present,
> not the night before.

## 1. Dependencies

In [ ]:
# Plain Python rather than `!pip`, so this cell is valid Python and the repo's
# notebook validator can parse it like every other cell.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "fastapi", "uvicorn[standard]", "transformers",
                "sentencepiece", "huggingface_hub"], check=True)
print("dependencies installed")

## 2. Sign in to Hugging Face

The model repository is private, so this needs a **read**-scoped token. The
prompt below hides what you type — nothing is written into the notebook, and
nothing is saved when the session ends.

In [ ]:
from huggingface_hub import login
login()

## 3. Fetch the app from GitHub

The same `app.py` and `index.html` that ran on the training node. Nothing is
duplicated here, so a fix pushed to the repository reaches this notebook on the
next run.

In [ ]:
import pathlib, urllib.request

BASE = ("https://raw.githubusercontent.com/SamAbr/"
        "PSA-MT/master")
FILES = ["serve/app.py",
         "serve/static/index.html",
         "serve/metrics/evaluation_results.csv"]

for f in FILES:
    pathlib.Path(f).parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{BASE}/{f}", f)
    print(f"{pathlib.Path(f).stat().st_size:>9,}  {f}")

## 4. Start it

Loads the released single-pass model, starts the API, and opens a Cloudflare
tunnel. The model download is the slow part; the tunnel takes seconds.

In [ ]:
import os, pathlib, platform, re, subprocess, time, urllib.request

os.environ["ENABLED_SYSTEMS"] = "mixed"
os.environ["MAX_RESIDENT"] = "1"
os.environ["PRELOAD"] = "1"
os.environ["RATE_LIMIT_PER_MIN"] = "30"
# Set this to a dataset repo you own to keep corrections. Without it they live
# only in this Colab session and vanish with it.
# os.environ["FEEDBACK_REPO"] = "samuelabrha/ekegusii-feedback"

PORT = 8000
cf = pathlib.Path("cloudflared")
if not cf.exists():
    arch = {"x86_64": "amd64", "aarch64": "arm64"}[platform.machine()]
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        f"cloudflared-linux-{arch}", cf)
    cf.chmod(0o755)
    print(f"cloudflared {cf.stat().st_size:,} bytes")

api = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd="serve", stdout=open("api.log", "wb"), stderr=subprocess.STDOUT)

print("loading the model (2.4 GB on the first run) ...")
for _ in range(600):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=3).read()
        break
    except Exception:
        if api.poll() is not None:
            print(open("api.log").read()[-3000:])
            raise SystemExit("the API died - see the log above")
        time.sleep(1)
else:
    raise SystemExit("the API never became healthy - check api.log")

print(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health").read().decode())

tunnel = subprocess.Popen(
    [str(cf.resolve()), "tunnel", "--no-autoupdate", "--protocol", "http2",
     "--url", f"http://127.0.0.1:{PORT}"],
    stdout=open("tunnel.log", "wb"), stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    hit = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                    pathlib.Path("tunnel.log").read_text(errors="ignore"))
    if hit:
        url = hit.group(0)
        break
    time.sleep(1)

if not url:
    print(pathlib.Path("tunnel.log").read_text()[-2000:])
    raise SystemExit("no tunnel URL appeared")

print("\n" + "=" * 62)
print(f"  {url}")
print("=" * 62)
print("  Keep this cell running. Closing the notebook closes the tunnel.")

## 5. Keep the session alive

Colab disconnects an idle notebook. Run this cell during a presentation and it
pings the service every few minutes, which both keeps Colab awake and confirms
the model is still answering.

Stop it with the square button when you're finished.

In [ ]:
import time, urllib.request

while True:
    try:
        body = urllib.request.urlopen(
            f"http://127.0.0.1:{PORT}/api/health", timeout=5).read().decode()
        print(time.strftime("%H:%M:%S"), body[:90])
    except Exception as exc:
        print(time.strftime("%H:%M:%S"), "UNHEALTHY:", exc)
    time.sleep(240)

## Troubleshooting

**401 on every translation** — the token in step 2 has no read access to
`samuelabrha/nllb-200-600M-ekegusii-mixed`, or you skipped step 2.

**Out of memory** — you are on a CPU runtime and something else is using the
RAM. Runtime → Change runtime type → T4 GPU.

**Tunnel never appears** — Colab's network occasionally blocks the tunnel. Run
step 4 again; a new URL is issued each time.

**The URL stopped working** — the Colab session ended. Re-run step 4. The URL
changes every time; never write it into a document.